# CardiaSense — Embeddings, product report & web app

**Educational technical walkthrough · notebook 4 of 5**

**Objectives:** encoder token embeddings, ensemble report internals, README audio/screenshots

**You will need:** the `cardiasense` venv (run `./setup.sh`), the dataset at
`$CARDIASENSE_DATA_DIR` (downloaded by `setup.sh`), and a trained head for the
notebooks that use checkpoints (or run the smoke train in notebook 03 first).

> Educational / research prototype — **not a medical device**.

---




### Step 1 — The encoder as a tokenizer (research direction A)

The AST-style transformer splits the spectrogram into **patches** and maps each
to a token; a `[CLS]` token pools the whole clip. `encode()` exposes these
internals: `(patch_tokens (T, d), cls (d))`.

**Hypothesis:** the encoder already *organises* these vectors so class
embeddings cluster — measure it below with the **cosine similarity between
class-mean CLS embeddings**. A sharp diagonal = classes separated in embedding
space → those vectors are a ready-made, ordered interface for a downstream LLM
(frozen encoder + linear projection, or patch tokens as soft prompts).

> If no transformer checkpoint is present this cell prints a note instead of
> failing — run `03`'s full campaign first to have one.


In [1]:

import os, sys
from pathlib import Path

ROOT = Path("/home/tnzr/CardiaSense-AI")
sys.path.insert(0, str(ROOT / "ml"))
sys.path.insert(0, str(ROOT))

from cardia.config import DEFAULT_DATA_DIR   # loads .env if present
DATA = os.environ.get("CARDIASENSE_DATA_DIR") or DEFAULT_DATA_DIR
print("Dataset root:", DATA)


Dataset root: /home/tnzr/CardiaSense-AI/data/HLS-CMDS


In [2]:

import matplotlib.pyplot as plt
from cardia.config import HEART_TYPES

MC = "/media/tnzr/AuxVolume/cardiasense-runs/mc"   # multiclass campaign (Transformer heads)
try:
    from cardia.features import embeddings as emb
    data = emb.corpus_embeddings(DATA, MC)
    M = emb.class_mean_matrix(data)
    print("clips:", len(data["rows"]["sample_id"]), "| embedding dim:", data["rows"]["cls"][0].shape)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(M, cmap="magma", vmin=0, vmax=1)
    ax.set_xticks(range(len(HEART_TYPES))); ax.set_yticks(range(len(HEART_TYPES)))
    ax.set_xticklabels([c[:10] for c in HEART_TYPES], rotation=45, ha="right")
    ax.set_yticklabels([c[:10] for c in HEART_TYPES])
    ax.set_title("Class-mean CLS embedding cosine similarity (encoder-organised)")
    plt.show()
except FileNotFoundError as e:
    print("Transformer checkpoint not available here:", e)


Transformer checkpoint not available here: [Errno 2] No such file or directory: '/media/tnzr/AuxVolume/cardiasense-runs/mc/splits/heart_10class_s0/fold0_train.csv'



### Step 2 — A product-style report in one call

The web app runs **5-fold ensembles** of the heads, sweeps the clip in a sliding
window, and renders a **synchronized inference timeline** (spectrogram on top,
stacked-area probabilities in the middle, confidence line at the bottom — all
time-aligned). This cell produces the same objects the report page renders, so
you can inspect the structure directly.


In [3]:

import sys, torch, soundfile as sf
sys.path.insert(0, str(ROOT / "webapp"))
from webapp.app import inference, report, game

reg = inference.get_registry()               # warm-loads head ensembles (CPU here)
opts = report.ReportOptions(heads=["source", "heart_types"], figures="all",
                            temporal=True, temporal_mode="multiscale", explanation="standard")
path = game.resolve_audio_path("M0001", "sound", "binary")
wav, sr = sf.read(path, dtype="float32", always_2d=True)
rep = report.analyze_waveform(torch.from_numpy(wav[:, 0]).squeeze(), int(sr), "M0001.wav", opts)
print("verdict:", rep["verdict"], "| heads used:", [h["key"] for h in rep["heads"]])
print("figures:", list(rep["figures"].keys())[:6], "...")


verdict: no-flag | heads used: ['source']
figures: ['waveform', 'overlay', 'spectrogram', 'probs_source', 'multiscale_source', 'timeline_source'] ...



### Step 3 — What ships in the repo

The GitHub README plays sample clips directly (HTML5 `<audio>` — GitHub renders
them from committed WAVs) and the walkthrough embeds web-app screenshots. The
commands below point you at those artifacts; `scripts/screenshot_walkthrough.py`
regenerates the screenshots and `scripts/make_audio_assets.py` the clips.


In [4]:

print("Audio assets:   docs/assets/audio/*.wav   (played inline in the README)")
print("Screenshots:    docs/walkthrough/*.png    (instructional walkthrough)")
print("Report:         ml/report/cardiasense_report.pdf (regenerate via cardia.report.report)")


Audio assets:   docs/assets/audio/*.wav   (played inline in the README)
Screenshots:    docs/walkthrough/*.png    (instructional walkthrough)
Report:         ml/report/cardiasense_report.pdf (regenerate via cardia.report.report)
